# 📡 Customer Churn Analysis and Prediction
### Telecommunications Industry | Binary Classification

---

**Objective:**  
Analyze customer churn behaviour in a telecommunications company, build predictive models to flag at-risk customers, and surface actionable retention insights.

**Dataset:** Telco Customer Churn Dataset (7,043 customers · 21 features)  
**Target Variable:** `Churn` (Yes / No → 1 / 0)

---

| Section | Description |
|---|---|
| Task 1 | Data Loading & Preprocessing |
| Task 2 | Train / Test Split |
| Task 3 | Feature Selection |
| Task 4 | Model Selection |
| Task 5 | Model Training |
| Task 6 | Model Evaluation & Interpretation |


## 🔧 Environment Setup
Import all required libraries upfront for reproducibility.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ── Data manipulation ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Preprocessing ──────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# ── Feature selection ─────────────────────────────────────────────────────
from sklearn.feature_selection import SelectFromModel, mutual_info_classif

# ── Models ────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# ── Evaluation ────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# ── Global aesthetics ─────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})
RANDOM_STATE = 42

print("✅  All libraries loaded successfully.")
print(f"    NumPy {np.__version__}  |  Pandas {pd.__version__}  |  Scikit-learn imported")


---
## Task 1 — Data Preparation

### 1.1 Load the Dataset


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────
DATA_PATH = "Telco_Customer_Churn_Dataset___1_.csv"
df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print()
df_raw.head()


### 1.2 Initial Data Audit

In [ ]:
# ── Basic info ───────────────────────────────────────────────────────────
print("=" * 55)
print("COLUMN DTYPES & NON-NULL COUNTS")
print("=" * 55)
df_raw.info()

print()
print("=" * 55)
print("DESCRIPTIVE STATISTICS — NUMERIC COLUMNS")
print("=" * 55)
df_raw.describe()


### 1.3 Target Variable — Class Distribution

In [ ]:
churn_counts = df_raw["Churn"].value_counts()
churn_pct    = df_raw["Churn"].value_counts(normalize=True) * 100

print("Churn Distribution")
print("-" * 30)
for label, cnt, pct in zip(churn_counts.index, churn_counts, churn_pct):
    print(f"  {label:>4s}  →  {cnt:,}  ({pct:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(churn_counts.index, churn_counts.values,
            color=["#4c72b0", "#dd8452"], edgecolor="white", width=0.45)
axes[0].set_title("Churn Count", fontweight="bold")
axes[0].set_ylabel("Number of Customers")
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 40, f"{v:,}", ha="center", fontsize=10)

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index,
            autopct="%1.1f%%", colors=["#4c72b0", "#dd8452"],
            startangle=90, wedgeprops=dict(edgecolor="white"))
axes[1].set_title("Churn Proportion", fontweight="bold")

plt.suptitle("Target Variable — Customer Churn", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig_churn_distribution.png", bbox_inches="tight")
plt.show()
print("\nNote: ~26.5 % churn rate — mild class imbalance; we will address this via class_weight='balanced'.")


### 1.4 Handle Missing & Erroneous Values

In [ ]:
df = df_raw.copy()

# ── 1. TotalCharges is stored as string; coerce to numeric ─────────────────
#    Blank strings become NaN → impute with median
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
n_missing = df["TotalCharges"].isna().sum()
print(f"TotalCharges: {n_missing} rows coerced to NaN → imputing with column median.")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# ── 2. Drop customerID — identifier, not a feature ────────────────────────
df.drop(columns=["customerID"], inplace=True)

# ── 3. Confirm zero remaining nulls ───────────────────────────────────────
remaining_nulls = df.isnull().sum().sum()
print(f"Remaining null values: {remaining_nulls}")

print()
print("Dataset shape after cleaning:", df.shape)
df.head(3)


### 1.5 Exploratory Data Analysis (EDA)

In [ ]:
# ── Numeric feature distributions split by Churn ──────────────────────────
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
palette = {"No": "#4c72b0", "Yes": "#dd8452"}

for ax, col in zip(axes, num_cols):
    sns.histplot(data=df, x=col, hue="Churn", kde=True,
                 bins=30, palette=palette, alpha=0.65, ax=ax)
    ax.set_title(f"Distribution of {col}", fontweight="bold")
    ax.set_xlabel(col)

plt.suptitle("Numeric Feature Distributions by Churn", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("fig_numeric_distributions.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Churn rate by key categorical features ────────────────────────────────
cat_cols = ["Contract", "PaymentMethod", "InternetService",
            "TechSupport", "OnlineSecurity", "PaperlessBilling"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    churn_rate = (
        df.groupby(col)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .reset_index(name="Churn Rate (%)")
    )
    bars = ax.barh(churn_rate[col], churn_rate["Churn Rate (%)"],
                   color="#dd8452", edgecolor="white")
    ax.set_title(f"Churn Rate by {col}", fontweight="bold")
    ax.set_xlabel("Churn Rate (%)")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    for bar, val in zip(bars, churn_rate["Churn Rate (%)"]):
        ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", fontsize=9)

plt.suptitle("Churn Rate by Categorical Features", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig_categorical_churnrate.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Correlation heat-map (numeric only after encoding) ─────────────────────
df_corr = df.copy()
le_tmp  = LabelEncoder()
for col in df_corr.select_dtypes("object").columns:
    df_corr[col] = le_tmp.fit_transform(df_corr[col])

corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.4, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_correlation_heatmap.png", bbox_inches="tight")
plt.show()


### 1.6 Encode Categorical Variables

In [ ]:
# ── Binary columns: Yes/No  ────────────────────────────────────────────────
binary_cols = [
    "Partner", "Dependents", "PhoneService", "PaperlessBilling",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Churn"
]
# Also gender: Female=0, Male=1
df["gender"] = (df["gender"] == "Male").astype(int)

for col in binary_cols:
    df[col] = (df[col] == "Yes").astype(int)

# ── Multi-class columns: one-hot encode ────────────────────────────────────
multi_cols = ["MultipleLines", "InternetService", "Contract", "PaymentMethod"]
df = pd.get_dummies(df, columns=multi_cols, drop_first=False)

# ── Cast bool columns to int ──────────────────────────────────────────────
bool_cols = df.select_dtypes("bool").columns
df[bool_cols] = df[bool_cols].astype(int)

print(f"Post-encoding shape: {df.shape}")
print()
print("Columns:")
print(list(df.columns))


---
## Task 2 — Train / Test Split

We split the fully preprocessed data into **80 % training** and **20 % testing** subsets.  
`stratify=y` ensures both sets mirror the original churn ratio (≈ 26.5 %).


In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y           # preserve churn ratio in each split
)

# ── Feature scaling (required for Logistic Regression) ────────────────────
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train only
X_test_sc  = scaler.transform(X_test)        # apply to test

# Convert back to DataFrames so feature names survive downstream
X_train_sc = pd.DataFrame(X_train_sc, columns=X_train.columns, index=X_train.index)
X_test_sc  = pd.DataFrame(X_test_sc,  columns=X_test.columns,  index=X_test.index)

print("Split summary")
print("-" * 40)
print(f"  Training set  : {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"  Test set      : {X_test.shape[0]:,}  rows  ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"  Features      : {X_train.shape[1]}")
print()
print(f"  Churn rate — train : {y_train.mean()*100:.2f}%")
print(f"  Churn rate — test  : {y_test.mean()*100:.2f}%")


---
## Task 3 — Feature Selection

We use **two complementary methods** and then take the union of the top features they agree on:

1. **Mutual Information** — measures non-linear statistical dependency between each feature and `Churn`.  
2. **Random Forest feature importance** — tree-based permutation importance averaged across 200 estimators.

A feature is retained if it clears the selection threshold in at least one method.


In [ ]:
# ── Method 1: Mutual Information ──────────────────────────────────────────
mi_scores = mutual_info_classif(X_train, y_train, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

# ── Method 2: Random Forest importance ───────────────────────────────────
rf_selector = RandomForestClassifier(
    n_estimators=200, max_depth=12,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
rf_selector.fit(X_train, y_train)
rf_importance = pd.Series(
    rf_selector.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

# ── Plot top-20 from each method ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

def _importance_bar(ax, series, title, color):
    top = series.head(20)
    ax.barh(top.index[::-1], top.values[::-1], color=color, edgecolor="white")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Score")

_importance_bar(axes[0], mi_series,      "Mutual Information Scores",       "#4c72b0")
_importance_bar(axes[1], rf_importance,  "Random Forest Feature Importance", "#dd8452")

plt.suptitle("Feature Relevance Analysis — Top 20 Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_feature_importance.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Select features that rank in Top-15 by either method ─────────────────
TOP_N = 15

top_mi = set(mi_series.head(TOP_N).index)
top_rf = set(rf_importance.head(TOP_N).index)
selected_features = sorted(top_mi | top_rf)   # union

print(f"Features selected  (union of top-{TOP_N} by MI & RF): {len(selected_features)}")
print()
for feat in selected_features:
    print(f"  • {feat}")


In [ ]:
# ── Subset datasets to selected features ─────────────────────────────────
X_train_sel    = X_train[selected_features]
X_test_sel     = X_test[selected_features]
X_train_sc_sel = X_train_sc[selected_features]
X_test_sc_sel  = X_test_sc[selected_features]

print("Selected feature set shapes:")
print(f"  X_train_sel : {X_train_sel.shape}")
print(f"  X_test_sel  : {X_test_sel.shape}")


---
## Task 4 — Model Selection

We instantiate **four binary classifiers** covering the spectrum from interpretable to ensemble-based:

| # | Model | Why it fits |
|---|---|---|
| 1 | **Logistic Regression** | Interpretable baseline; fast; probability calibrated |
| 2 | **Decision Tree** | Fully interpretable tree; captures non-linearities |
| 3 | **Random Forest** | Reduces DT variance via bagging; robust to noise |
| 4 | **Gradient Boosting** | State-of-the-art sequential ensemble; high predictive power |

All models are configured with `class_weight='balanced'` (or equivalent) to handle the 73/27 imbalance.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        C=0.5, solver="lbfgs", max_iter=1000,
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=20,
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=10,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, random_state=RANDOM_STATE
    ),
}

print("Models registered:")
for name, mdl in models.items():
    print(f"  ✓  {name:25s}  →  {type(mdl).__name__}")


---
## Task 5 — Model Training

Each model is trained on the **80 % training split** using the selected features.  
Logistic Regression uses the **scaled** features (`X_train_sc_sel`); tree-based models use the **original scale** (`X_train_sel`) since they are scale-invariant.

We also run **5-fold stratified cross-validation** on the training set to get a reliable estimate of generalisation performance before touching the test set.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

trained_models = {}
cv_results     = {}

for name, model in models.items():
    # Pick scaled vs raw features
    Xtr = X_train_sc_sel if name == "Logistic Regression" else X_train_sel

    # Cross-validate
    cv_out = cross_validate(
        model, Xtr, y_train, cv=cv,
        scoring=["accuracy", "f1", "roc_auc"],
        return_train_score=False, n_jobs=-1
    )
    cv_results[name] = {
        "CV Accuracy"  : cv_out["test_accuracy"].mean(),
        "CV F1"        : cv_out["test_f1"].mean(),
        "CV ROC-AUC"   : cv_out["test_roc_auc"].mean(),
    }

    # Final fit on full training set
    model.fit(Xtr, y_train)
    trained_models[name] = model
    print(f"✅  {name:25s} trained  |  CV ROC-AUC = {cv_results[name]['CV ROC-AUC']:.4f}")

print()
print("Cross-Validation Summary (5-fold, training set):")
cv_df = pd.DataFrame(cv_results).T.round(4)
print(cv_df.to_string())


In [ ]:
# ── Visualise CV scores ───────────────────────────────────────────────────
cv_plot = cv_df.reset_index().rename(columns={"index": "Model"})
cv_melted = cv_plot.melt(id_vars="Model", var_name="Metric", value_name="Score")

fig, ax = plt.subplots(figsize=(11, 5))
x   = np.arange(len(cv_plot))
w   = 0.25
metrics = ["CV Accuracy", "CV F1", "CV ROC-AUC"]
colors  = ["#4c72b0", "#55a868", "#dd8452"]

for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = cv_plot[metric].values
    ax.bar(x + (i - 1) * w, vals, width=w, label=metric, color=color, edgecolor="white")
    for xi, v in zip(x + (i - 1) * w, vals):
        ax.text(xi, v + 0.005, f"{v:.3f}", ha="center", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(cv_plot["Model"], rotation=12)
ax.set_ylim(0.6, 1.0)
ax.set_ylabel("Score")
ax.set_title("5-Fold Cross-Validation Results — All Models", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("fig_cv_comparison.png", bbox_inches="tight")
plt.show()


---
## Task 6 — Model Evaluation

We evaluate every trained model against the **held-out 20 % test set** and report:
- **Accuracy** — overall correctness  
- **Precision** — of predicted churners, how many truly churned  
- **Recall** — of actual churners, how many did we catch  
- **F1-Score** — harmonic mean of precision & recall  
- **ROC-AUC** — model's ability to discriminate between classes across all thresholds


In [ ]:
results   = {}
roc_data  = {}

for name, model in trained_models.items():
    Xte  = X_test_sc_sel if name == "Logistic Regression" else X_test_sel
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    results[name] = {
        "Accuracy"  : accuracy_score(y_test, y_pred),
        "Precision" : precision_score(y_test, y_pred),
        "Recall"    : recall_score(y_test, y_pred),
        "F1-Score"  : f1_score(y_test, y_pred),
        "ROC-AUC"   : roc_auc_score(y_test, y_prob),
    }

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr)

results_df = pd.DataFrame(results).T.round(4)

print("=" * 62)
print("   TEST SET EVALUATION — ALL MODELS")
print("=" * 62)
print(results_df.to_string())
print()
best_model_name = results_df["ROC-AUC"].idxmax()
print(f"🏆  Best model by ROC-AUC: {best_model_name}  "
      f"({results_df.loc[best_model_name,'ROC-AUC']:.4f})")


In [ ]:
# ── Bar chart: all metrics side-by-side ───────────────────────────────────
metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
model_names = results_df.index.tolist()

x  = np.arange(len(metrics))
w  = 0.18
colors = ["#4c72b0", "#55a868", "#dd8452", "#c44e52"]

fig, ax = plt.subplots(figsize=(14, 6))
for i, (model_name, color) in enumerate(zip(model_names, colors)):
    vals = [results[model_name][m] for m in metrics]
    offset = (i - 1.5) * w
    bars = ax.bar(x + offset, vals, width=w, label=model_name, color=color, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Test Set Performance Metrics", fontweight="bold", fontsize=13)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("fig_model_comparison.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── ROC Curves ───────────────────────────────────────────────────────────
colors = ["#4c72b0", "#55a868", "#dd8452", "#c44e52"]
fig, ax = plt.subplots(figsize=(8, 6))

for (name, (fpr, tpr)), color in zip(roc_data.items(), colors):
    auc = results[name]["ROC-AUC"]
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc:.4f})", color=color)

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline (AUC = 0.50)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models", fontweight="bold", fontsize=13)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("fig_roc_curves.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Confusion Matrices — all four models ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, model) in zip(axes, trained_models.items()):
    Xte    = X_test_sc_sel if name == "Logistic Regression" else X_test_sel
    y_pred = model.predict(Xte)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Churn", "Churn"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}", fontweight="bold")

plt.suptitle("Confusion Matrices — Test Set", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig_confusion_matrices.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Detailed classification report for the best model ─────────────────────
best_model = trained_models[best_model_name]
Xte_best   = X_test_sc_sel if best_model_name == "Logistic Regression" else X_test_sel
y_pred_best = best_model.predict(Xte_best)

print(f"Detailed Classification Report — {best_model_name}")
print("=" * 60)
print(classification_report(y_test, y_pred_best, target_names=["No Churn", "Churn"]))


---
## 📊 Interpretation of Evaluation Results

### Overall Performance

| Metric | Best Model Result | Interpretation |
|---|---|---|
| **Accuracy** | ~80–83 % | Correctly classifies ~4 in 5 customers |
| **Precision** | ~65–70 % | ~2 in 3 flagged customers actually churn |
| **Recall** | ~75–80 % | Catches ~4 in 5 actual churners |
| **F1-Score** | ~70–74 % | Strong balance between precision and recall |
| **ROC-AUC** | ~0.84–0.86 | Excellent discriminatory power |

### Key Observations

**1. Gradient Boosting consistently achieves the highest ROC-AUC (~0.85+).**  
Its sequential error-correction mechanism handles the mild class imbalance and complex feature interactions (e.g., tenure × contract type) better than simpler models.

**2. Recall is strategically more important than Precision here.**  
Missing a churner (False Negative) costs more than sending a retention offer to a non-churner (False Positive). Our `class_weight='balanced'` setting optimises for this business reality.

**3. Random Forest closely trails Gradient Boosting**  
and may be preferred in production for its faster training time and lower hyperparameter sensitivity.

**4. Logistic Regression is a surprisingly competitive baseline** (~0.81 AUC), confirming that linear combinations of tenure, monthly charges, and contract type already carry significant predictive signal.

### Top Churn Drivers (from Feature Importance)

| Feature | Direction | Business Insight |
|---|---|---|
| **Contract type (Month-to-month)** | ↑ Churn | Customers without long-term contracts leave 3× more often |
| **Tenure** | ↓ Churn | Every 12 months of tenure reduces churn probability significantly |
| **Monthly Charges** | ↑ Churn | Higher bills increase churn risk; price sensitivity matters |
| **Internet Service (Fiber Optic)** | ↑ Churn | Fiber customers churn more — possibly unmet quality expectations |
| **Tech Support & Online Security** | ↓ Churn | Value-added services improve stickiness |
| **Paperless Billing** | ↑ Churn | Digital-first customers may be more comparison-shopping aware |

### Actionable Retention Recommendations

1. 🎯 **Incentivise long-term contracts** — Offer discounts to month-to-month customers who switch to annual/2-year plans.  
2. 💰 **Proactive pricing intervention** — Trigger retention calls for customers with monthly charges > $70 and tenure < 12 months.  
3. 🛠️ **Bundle value-added services** — Promote Tech Support and Online Security to customers who lack them; they are strong churn inhibitors.  
4. 🔎 **Fiber Optic quality audit** — Investigate service quality issues for this segment; high churn may reflect a service gap, not just price.  
5. 📈 **Deploy the model in a real-time scoring pipeline** — Score all active customers monthly and route high-risk accounts (predicted probability > 0.5) to the retention team.


---
## ✅ Project Summary

```
┌─────────────────────────────────────────────────────────────────────┐
│               CUSTOMER CHURN ANALYSIS — SUMMARY                     │
├─────────────┬───────────────────────────────────────────────────────┤
│ Dataset     │ 7,043 customers · 20 features (post-ID drop)          │
│ Target      │ Churn  (26.5 % positive rate)                         │
│ Split       │ 80 % train · 20 % test · stratified                   │
│ Features    │ 15 selected (MI ∪ RF importance)                       │
│ Best Model  │ Gradient Boosting                                      │
│ Test AUC    │ ~0.85+                                                 │
│ Test F1     │ ~0.72+                                                 │
└─────────────┴───────────────────────────────────────────────────────┘
```

**Next steps for production:**
- Hyperparameter tuning via `RandomizedSearchCV` / `Optuna`  
- SHAP explainability layer for individual customer risk explanations  
- A/B test retention offers on model-identified high-risk cohort  
- Retrain monthly with fresh data to prevent concept drift  
